In [2]:
3+5

8

In [6]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-120b")
llm.invoke("hi")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user just says "hi". We need to respond politely. No special instructions. Should greet back.'}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 72, 'total_tokens': 112, 'completion_time': 0.083553915, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.003333573, 'prompt_tokens_details': None, 'queue_time': 0.246882237, 'total_time': 0.086887488}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b1dd3e7a63', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1c58a-9b72-7465-bb91-a61fac08e615-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 40, 'total_tokens': 112, 'output_token_details': {'reasoning': 22}})

In [7]:
from langchain.tools import tool
@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two integers.

    Args:
        a (int): The first integer.
        b (int): The second integer.

    Returns:
        int: The product of a and b.
    """
    return a * b

In [9]:
multiply

StructuredTool(name='multiply', description='Multiply two integers.\n\nArgs:\n    a (int): The first integer.\n    b (int): The second integer.\n\nReturns:\n    int: The product of a and b.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000002AE91BC9F30>)

In [ ]:
from langchain_core.tools import StructuredTool

from pydantic import BaseModel # this is for validation that weather city is str ot not 
class WeatherInput(BaseModel):
    city: str

def get_weather(city: str) -> str:
    """
    Get the weather for a given city.

    Args:
        city (str): The name of the city.

    Returns:
        str: A string describing the weather in the city.
    """
    return f"The weather in {city} is sunny."

In [ ]:
# this way allows us to have more authority on tools
weather_tool = StructuredTool.from_function( # calling StructuredTool.from_function
    func=get_weather, # this is our function, could be any function
    name="get_weather",
    description="Fetches real-time weather data for a city",
    args_schema=WeatherInput, # validation by pydantic BaseModel defined above  
)

In [ ]:
# another approach to capture everything in class
# 3rd approachc
class WeatherInput(BaseModel):
    city: str = Field(..., description="City name")
    units: str = Field("metric", description="metric or imperial")

class GetWeatherTool(StructuredTool):
    name: ClassVar[str] = "get_weather" # now it will be considered as tool  
    description: ClassVar[str] = (
        "Fetches weather data for a city"
    )
    args_schema: ClassVar[Type[BaseModel]] = WeatherInput

    def _run(self, city: str, units: str = "metric") -> str:
        return get_weather(city, units)